In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
df = pd.read_excel("/kaggle/input/datasets/anvarnarz/tashkent-real-estate-2019/uybor.xlsx")
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print(df.shape)
df.info()
print(df.isnull().sum())
print("Takroriy qatorlar:", df.duplicated().sum())
df.describe()

In [ ]:
df = df.drop_duplicates().dropna(subset=["price", "size", "rooms", "district"])

# Eng yuqori va eng past 1% ni kesib tashlaymiz
for col in ["price", "size"]:
    low, high = df[col].quantile([0.01, 0.99])
    df = df[(df[col] >= low) & (df[col] <= high)]

df["price_per_m2"] = df["price"] / df["size"]
print(df.shape)

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df["price"], bins=40, kde=True)
plt.title("Uylar narxi taqsimoti")
plt.show()

In [ ]:
district_price = df.groupby("district")["price"].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=district_price.values, y=district_price.index)
plt.title("Tumanlar bo'yicha o'rtacha narx")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="size", y="price", hue="rooms", palette="viridis")
plt.title("Maydon va narx munosabati")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[["rooms", "size", "level", "max_levels", "price"]].corr(), annot=True, cmap="coolwarm")
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sc = plt.scatter(df["lng"], df["lat"], c=df["price_per_m2"], cmap="plasma", s=10)
plt.colorbar(sc, label="1 m² narxi")
plt.title("Toshkentda 1 m² narxi xaritasi")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X = df[["rooms", "size", "level", "max_levels"]]
y = df["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression().fit(X_train, y_train)
print("R2:", r2_score(y_test, model.predict(X_test)))

In [ ]:
print(district_price.round(0).head(3))
print(district_price.round(0).tail(3))
print(df[["rooms", "size", "level", "max_levels", "price"]].corr()["price"].round(2))
print("O'rtacha 1 m2 narxi:", round(df["price_per_m2"].mean(), 1))
print("Medianali narx:", df["price"].median())

## Xulosalar

1. **Narx taqsimoti:** narxlar o'ng tomonga cho'zilgan. Ko'pchilik uylar arzon va o'rta segmentda, kam sonli uylar juda qimmat. Butun uyning mediana narxi 46 500, 1 m² ning o'rtacha narxi esa taxminan 775.
2. **Tumanlar:** eng qimmat tumanlar **Mirobod** (o'rtacha 84 756), **Yakkasaroy** (67 722) va **Shayxontohur** (64 405). Eng arzon tumanlar **Bektemir** (29 462), **Yangihayot** (37 192) va **Sergeli** (44 843). Mirobod Bektemirdan taxminan 2.9 baravar qimmat.
3. **Omillar:** narx bilan eng kuchli bog'liqlik **maydonda** (korrelyatsiya 0.79), keyin **xonalar soni** (0.56) va uy qavatliligi (0.25). Uy qavati (0.07) deyarli ta'sir qilmaydi.
4. **Xarita:** 1 m² narxi joylashuvga qarab farq qiladi, markaziy hududlarda yuqori.
5. **Model:** chiziqli regressiya R² = 0.65 natija berdi, ya'ni narx o'zgarishining taxminan 65% ini tushuntiradi. Tuman va koordinatalarni qo'shsak, aniqlik oshishi mumkin.
6. **Cheklov:** ma'lumotlar 2019 yilga tegishli va uybor.uz e'lonlaridan olingan, shuning uchun hozirgi bozor narxlarini to'liq aks ettirmaydi.
7. **Keyingi qadam:** `district` ustunini modelga qo'shib, aniqlikni oshirish mumkin.